# ml_training/notebooks/EDA_solar_wind.ipynb

# NAKSHATRA-KAVACH Layer 2 EDA

Exploratory analysis for the 45-feature solar-wind engineering pipeline, including distributions, correlation structure, feature-vs-Kp relationships, rolling-window behavior, missing data, and the May 2024 G5 storm case study.

In [1]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT.name != 'Internal Hackathon 2026' and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent
BACKEND_ROOT = PROJECT_ROOT / 'backend'
if str(BACKEND_ROOT) not in sys.path:
    sys.path.insert(0, str(BACKEND_ROOT))

from app.services.feature_engineering import FEATURE_NAMES

OUTPUT_DIR = PROJECT_ROOT / 'ml_training' / 'output'
sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.figsize'] = (12, 6)

## Load Engineered Training Outputs

In [2]:
def load_split(name: str):
    x_path = OUTPUT_DIR / f'X_{name}.csv'
    y_path = OUTPUT_DIR / f'y_{name}.csv'
    X = pd.read_csv(x_path, parse_dates=['timestamp_utc']).set_index('timestamp_utc')
    y = pd.read_csv(y_path, parse_dates=['timestamp_utc']).set_index('timestamp_utc')
    return X, y

X_train, y_train = load_split('train')
X_val, y_val = load_split('val')
X_test, y_test = load_split('test')
X_all = pd.concat([X_train, X_val, X_test]).sort_index()
y_all = pd.concat([y_train, y_val, y_test]).sort_index()
display(X_all.head())
display(y_all.describe())

FileNotFoundError: [Errno 2] No such file or directory: 'c:\\Users\\saina\\Videos\\Internal Hackathon 2026\\ml_training\\output\\X_train.csv'

## Distribution Plots For Each Feature

In [ ]:
def plot_feature_distributions(X: pd.DataFrame, columns=None, bins: int = 60):
    columns = list(columns or X.columns)
    ncols = 3
    nrows = int(np.ceil(len(columns) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(18, max(4, nrows * 3)))
    axes = np.asarray(axes).reshape(-1)
    for ax, col in zip(axes, columns):
        sns.histplot(X[col].dropna(), bins=bins, kde=True, ax=ax, color='#246B73')
        ax.set_title(col)
    for ax in axes[len(columns):]:
        ax.axis('off')
    fig.tight_layout()
    return fig

plot_feature_distributions(X_all[FEATURE_NAMES])

## Correlation Matrix Heatmap

In [ ]:
corr = X_all[FEATURE_NAMES].join(y_all['kp_3hr']).corr(numeric_only=True)
plt.figure(figsize=(18, 14))
sns.heatmap(corr, cmap='vlag', center=0, square=False, linewidths=0.1)
plt.title('Layer 2 Feature Correlation Matrix')
plt.show()

corr['kp_3hr'].drop('kp_3hr').abs().sort_values(ascending=False).head(20)

## Feature Vs Kp Scatter Plots

In [ ]:
def plot_feature_vs_kp(features, target='kp_3hr', sample=15000):
    data = X_all[list(features)].join(y_all[target]).dropna()
    if len(data) > sample:
        data = data.sample(sample, random_state=42)
    ncols = 3
    nrows = int(np.ceil(len(features) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(18, max(4, nrows * 4)))
    axes = np.asarray(axes).reshape(-1)
    for ax, feature in zip(axes, features):
        sns.scatterplot(data=data, x=feature, y=target, s=10, alpha=0.35, ax=ax, color='#8A4F2A')
        sns.regplot(data=data, x=feature, y=target, scatter=False, lowess=True, ax=ax, color='#1B1B1B')
        ax.set_title(f'{feature} vs {target}')
    for ax in axes[len(features):]:
        ax.axis('off')
    fig.tight_layout()
    return fig

plot_feature_vs_kp(['bz_min_1hr', 'bz_mean_1hr', 'epsilon_mean_1hr', 'epsilon_cumulative_3hr', 'sw_speed_mean_1hr', 'dynamic_pressure_max_1hr', 'consecutive_southward_minutes', 'kp_current', 'cme_is_imminent'])

## Rolling Window Visualization

In [ ]:
def plot_rolling_window(start, end):
    window = X_all.loc[start:end].join(y_all['kp_3hr'])
    fig, axes = plt.subplots(4, 1, figsize=(16, 12), sharex=True)
    window[['bz_current', 'bz_mean_30min', 'bz_mean_1hr', 'bz_mean_3hr']].plot(ax=axes[0])
    axes[0].axhline(-5, color='black', linestyle='--', linewidth=1)
    axes[0].set_ylabel('Bz nT')
    window[['epsilon_current', 'epsilon_mean_1hr']].plot(ax=axes[1])
    axes[1].set_ylabel('Epsilon scaled')
    window[['southward_fraction_30min', 'southward_fraction_1hr', 'consecutive_southward_minutes']].plot(ax=axes[2])
    axes[2].set_ylabel('Persistence')
    window[['kp_current', 'kp_3hr']].plot(ax=axes[3])
    axes[3].set_ylabel('Kp')
    fig.suptitle(f'Rolling Feature Evolution: {start} to {end}')
    fig.tight_layout()
    return fig

plot_rolling_window('2024-05-09', '2024-05-13')

## Missing Data Analysis

In [ ]:
missing = X_all.isna().mean().sort_values(ascending=False)
display(missing.head(20))
plt.figure(figsize=(14, 4))
missing.plot(kind='bar', color='#246B73')
plt.ylabel('Missing Fraction')
plt.title('Layer 2 Engineered Feature Missingness')
plt.tight_layout()
plt.show()
assert X_all[FEATURE_NAMES].isna().sum().sum() == 0, 'Layer 2 output must not contain NaN values'

## May 2024 G5 Storm Case Study

In [ ]:
def reproduce_may2024_storm():
    storm = X_all.loc['2024-05-10':'2024-05-12'].join(y_all)
    quiet = X_all.loc['2024-04-20':'2024-04-22'].join(y_all)
    if storm.empty:
        raise ValueError('May 10-12, 2024 storm window is not present in the engineered output')
    peak_time = storm['kp_3hr'].idxmax()
    quiet_time = quiet['kp_3hr'].idxmin() if not quiet.empty else X_all.index[0]
    compare_features = [
        'bz_current', 'bz_min_1hr', 'bz_mean_1hr', 'epsilon_current',
        'epsilon_mean_1hr', 'epsilon_cumulative_3hr', 'sw_speed_mean_1hr',
        'dynamic_pressure_max_1hr', 'consecutive_southward_minutes',
        'southward_fraction_1hr', 'kp_current', 'kp_3hr'
    ]
    comparison = pd.DataFrame({
        'quiet_baseline': X_all.join(y_all).loc[quiet_time, compare_features],
        'may2024_peak': X_all.join(y_all).loc[peak_time, compare_features],
    })
    display(comparison)
    plot_rolling_window('2024-05-09', '2024-05-13')
    return comparison

may2024_comparison = reproduce_may2024_storm()